In [1]:
# =========================================
# Normalization-controlled re-evaluation
# Raw vs Normalized Persian text
# =========================================

import os
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

# =========================================
# 1) Load data
# =========================================

DATA_PATH = "/kaggle/input/datasets/aabdollahii/humanvsai/dataset (1).xlsx"
assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH}"

df = pd.read_excel(DATA_PATH)
print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())

# =========================================
# 2) Basic cleaning
# =========================================

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[\u200b-\u200f\uFEFF]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# =========================================
# 3) Persian normalization
# =========================================

def normalize_persian(text):
    if pd.isna(text):
        return ""
    text = str(text)

    # Arabic to Persian characters
    text = text.replace("ي", "ی")
    text = text.replace("ى", "ی")
    text = text.replace("ك", "ک")
    text = text.replace("ؤ", "و")
    text = text.replace("ة", "ه")
    text = text.replace("ۀ", "ه")
    text = text.replace("أ", "ا")
    text = text.replace("إ", "ا")
    text = text.replace("ٱ", "ا")

    # Remove diacritics
    text = re.sub(r"[\u064B-\u065F\u0670\u06D6-\u06ED]", "", text)

    # Remove tatweel
    text = text.replace("ـ", "")

    # Normalize ZWNJ and spaces
    text = text.replace("\u200c", " ")
    text = text.replace("\u200f", " ")
    text = text.replace("\u200e", " ")

    # Normalize punctuation variants
    text = text.replace("،", ",")
    text = text.replace("؛", ";")
    text = text.replace("؟", "?")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")

    # Remove repeated punctuation
    text = re.sub(r"([,;:.!?]){2,}", r"\1", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

# =========================================
# 4) Build paired long-format dataset
# =========================================

df = df.reset_index().rename(columns={"index": "group_id"})

df["text_raw"] = df["text"].apply(clean_text)
df["machine_text_raw"] = df["machine_text"].apply(clean_text)

df["text_norm"] = df["text_raw"].apply(normalize_persian)
df["machine_text_norm"] = df["machine_text_raw"].apply(normalize_persian)

df = df[
    (df["text_raw"].str.len() > 0) &
    (df["machine_text_raw"].str.len() > 0)
].reset_index(drop=True)

def make_long_format(dataframe, human_col, machine_col):
    human_part = dataframe[["group_id", "year", "filename", human_col, "label"]].copy()
    human_part = human_part.rename(columns={human_col: "text", "label": "target"})
    human_part["source_type"] = "human"

    machine_part = dataframe[["group_id", "year", "filename", machine_col, "machine_label"]].copy()
    machine_part = machine_part.rename(columns={machine_col: "text", "machine_label": "target"})
    machine_part["source_type"] = "machine"

    long_df = pd.concat([human_part, machine_part], ignore_index=True)
    long_df = long_df[long_df["text"].astype(str).str.len() > 0].reset_index(drop=True)
    return long_df

df_long_raw = make_long_format(df, "text_raw", "machine_text_raw")
df_long_norm = make_long_format(df, "text_norm", "machine_text_norm")

print("Raw long shape:", df_long_raw.shape)
print("Normalized long shape:", df_long_norm.shape)

# =========================================
# 5) Encode labels
# =========================================

le = LabelEncoder()
y_raw = le.fit_transform(df_long_raw["target"].astype(str).values)
y_norm = le.transform(df_long_norm["target"].astype(str).values)

X_raw = df_long_raw["text"].astype(str).values
X_norm = df_long_norm["text"].astype(str).values

groups_raw = df_long_raw["group_id"].values
groups_norm = df_long_norm["group_id"].values

print("Label mapping:")
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    print(f"  {cls} -> {idx}")

positive_class = 1

# =========================================
# 6) Models
# =========================================

experiments = {
    "word_uni_bigram_svc": Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            lowercase=False
        )),
        ("clf", CalibratedClassifierCV(
            estimator=LinearSVC(C=1.0),
            method="sigmoid",
            cv=3
        ))
    ]),

    "charwb_3_5_svc": Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            lowercase=False
        )),
        ("clf", CalibratedClassifierCV(
            estimator=LinearSVC(C=1.0),
            method="sigmoid",
            cv=3
        ))
    ]),

    "word_uni_bigram_logreg": Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            lowercase=False
        )),
        ("clf", LogisticRegression(
            max_iter=3000,
            random_state=42
        ))
    ]),
}

# =========================================
# 7) Evaluation
# =========================================

def evaluate_cv(model, X, y, groups, n_splits=5, random_state=42):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    rows = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        clf = clone(model)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        y_score = None
        if hasattr(clf, "predict_proba"):
            y_score = clf.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_pred,
            average="binary",
            pos_label=positive_class,
            zero_division=0
        )

        roc_auc = np.nan
        if y_score is not None and len(np.unique(y_test)) == 2:
            try:
                roc_auc = roc_auc_score(y_test, y_score)
            except Exception:
                pass

        rows.append({
            "fold": fold,
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "roc_auc": roc_auc
        })

    fold_df = pd.DataFrame(rows)

    summary = {
        "accuracy_mean": fold_df["accuracy"].mean(),
        "accuracy_std": fold_df["accuracy"].std(ddof=1),
        "precision_mean": fold_df["precision"].mean(),
        "precision_std": fold_df["precision"].std(ddof=1),
        "recall_mean": fold_df["recall"].mean(),
        "recall_std": fold_df["recall"].std(ddof=1),
        "f1_mean": fold_df["f1"].mean(),
        "f1_std": fold_df["f1"].std(ddof=1),
        "roc_auc_mean": fold_df["roc_auc"].mean(skipna=True),
        "roc_auc_std": fold_df["roc_auc"].std(ddof=1, skipna=True)
    }

    return fold_df, summary

# =========================================
# 8) Run raw vs normalized comparison
# =========================================

all_results = []

for setting_name, X_data, y_data, groups_data in [
    ("raw", X_raw, y_raw, groups_raw),
    ("normalized", X_norm, y_norm, groups_norm),
]:
    print("\n" + "=" * 90)
    print("SETTING:", setting_name.upper())
    print("=" * 90)

    for exp_name, model in experiments.items():
        print("\n" + "-" * 90)
        print("Experiment:", exp_name)
        print("-" * 90)

        fold_df, summary = evaluate_cv(model, X_data, y_data, groups_data, n_splits=5)
        print(fold_df.round(4))

        row = {
            "setting": setting_name,
            "experiment": exp_name,
        }
        row.update(summary)
        all_results.append(row)

        print("\nSummary:")
        for k, v in summary.items():
            print(f"{k}: {v:.4f}" if pd.notna(v) else f"{k}: NaN")

results_df = pd.DataFrame(all_results)

# =========================================
# 9) Create before/after comparison table
# =========================================

compare_cols = [
    "accuracy_mean", "accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "f1_mean", "f1_std",
    "roc_auc_mean", "roc_auc_std"
]

raw_df = results_df[results_df["setting"] == "raw"].copy()
norm_df = results_df[results_df["setting"] == "normalized"].copy()

merged = raw_df.merge(
    norm_df,
    on="experiment",
    suffixes=("_raw", "_norm")
)

for metric in ["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "roc_auc_mean"]:
    merged[f"{metric}_drop"] = merged[f"{metric}_raw"] - merged[f"{metric}_norm"]

final_cols = [
    "experiment",
    "accuracy_mean_raw", "accuracy_mean_norm", "accuracy_mean_drop",
    "precision_mean_raw", "precision_mean_norm", "precision_mean_drop",
    "recall_mean_raw", "recall_mean_norm", "recall_mean_drop",
    "f1_mean_raw", "f1_mean_norm", "f1_mean_drop",
    "roc_auc_mean_raw", "roc_auc_mean_norm", "roc_auc_mean_drop"
]

final_table = merged[final_cols].sort_values("f1_mean_drop", ascending=False).reset_index(drop=True)

print("\n" + "=" * 90)
print("RAW VS NORMALIZED COMPARISON")
print("=" * 90)
print(final_table.round(4))






Original shape: (1519, 10)
Columns: ['year', 'filename', 'text', 'word_count', 'label', 'machine_text', 'machine_label', 'generation_status', 'generation_error', 'processed_at']
Raw long shape: (3038, 6)
Normalized long shape: (3038, 6)
Label mapping:
  human -> 0
  machine -> 1

SETTING: RAW

------------------------------------------------------------------------------------------
Experiment: word_uni_bigram_svc
------------------------------------------------------------------------------------------
   fold  accuracy  precision  recall      f1  roc_auc
0     1    0.9885        1.0  0.9770  0.9884   0.9992
1     2    0.9951        1.0  0.9901  0.9950   0.9977
2     3    0.9918        1.0  0.9836  0.9917   0.9947
3     4    0.9884        1.0  0.9769  0.9883   0.9961
4     5    0.9901        1.0  0.9803  0.9900   1.0000

Summary:
accuracy_mean: 0.9908
accuracy_std: 0.0028
precision_mean: 1.0000
precision_std: 0.0000
recall_mean: 0.9816
recall_std: 0.0055
f1_mean: 0.9907
f1_std: 0.0028